# 07 — Donor Static-Embedding Validation (the SALT-relevant question, done right)

**Corrected understanding from nb06:** ViDeBERTa embeddings load FINE (real trained `weight`,
std 0.1276). No random-init bug. ViDeBERTa = DeBERTa-V3, so the checkpoint carries TWO
embedding tensors via **GDES** (`weight` = generator/shared, `_weight` = the trained delta).
Both scored ~chance on kNN-overlap. But kNN-overlap@10 is a *harsh* metric — it needs exact
neighbor-set matches and reports weak-but-real structure as zero.

**Why this matters for SALT:** SALT reconstructs a target token as a linear combo of anchor
tokens *in the donor's static space*, then reuses those weights on NeoBERT anchors. That only
yields meaningful NeoBERT embeddings if the donor's static space is semantically organized.
So we must know, with sensitive + independent tests, whether it is.

**Tests (predictions registered in section D):**
| # | Test | If ViDeBERTa static IS semantic | If it is NOT |
|---|------|--------------------------------|--------------|
| A | GDES resolution: kNN soundness for `weight`, `_weight`, `weight+_weight` | one variant > 0.2 | all ~chance |
| B | Spearman corr (donor-cos vs FastText-cos, 40k pairs) — sensitive | ρ > 0.2 | ρ ≈ 0 |
| C | related-vs-random donor cosine gap | gap > 0.03 | gap ≈ 0 |

All three are also run for **PhoBERT** and the **NeoBERT-EN positive control**, so the verdict
is comparative, not absolute. DeBERTa-v3's RTD objective is known (ELECTRA family) to organize
static embeddings differently than MLM — this notebook decides if that sinks it as a SALT donor.


In [3]:
%%capture
!pip install -U transformers safetensors huggingface_hub fasttext-wheel scipy xformers


In [4]:
import sys
from pathlib import Path
import numpy as np, torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import importlib, salt3_diagnostics as dx, salt3_donor_analysis as da
importlib.reload(dx); importlib.reload(da)
from salt3_common import extract_embedding_weight
print('transformers', transformers.__version__, '| torch', torch.__version__)

FT_VI = str(PROJECT_ROOT / 'init' / 'videberta_salt_init_v5_globalmap_freqbias' / 'cc.vi.300.bin')
FT_EN = '/content/cc.en.300.bin'

vide_tok = AutoTokenizer.from_pretrained('Fsoft-AIC/videberta-base')
vide_model = AutoModel.from_pretrained('Fsoft-AIC/videberta-base', trust_remote_code=True)
vide_weight = extract_embedding_weight(vide_model).float().cpu()       # the loaded `weight`
vide_vocab = vide_tok.get_vocab()

pho_tok = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
pho_emb = extract_embedding_weight(AutoModel.from_pretrained('vinai/phobert-base-v2')).float().cpu()
pho_vocab = pho_tok.get_vocab()

neo_tok = AutoTokenizer.from_pretrained('chandar-lab/NeoBERT', trust_remote_code=True)
neo_emb = extract_embedding_weight(AutoModelForMaskedLM.from_pretrained(
    'chandar-lab/NeoBERT', trust_remote_code=True)).float().cpu()
neo_vocab = neo_tok.get_vocab()
print('embeddings ready')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
transformers 5.10.2 | torch 2.11.0+cu128


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: Fsoft-AIC/videberta-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/981M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

embeddings ready


## A. GDES resolution — which ViDeBERTa tensor (if any) is semantic?
Pull `weight` and `_weight` from the raw checkpoint, build `weight + _weight`, and score all
three. If the correct discriminator embedding is the sum (GDES: E_D = E_G + E_Δ), it should
beat either component.


In [5]:
import fasttext
from huggingface_hub import hf_hub_download
ft_vi = fasttext.load_model(FT_VI)

sd = torch.load(hf_hub_download('Fsoft-AIC/videberta-base', 'pytorch_model.bin'),
                map_location='cpu', weights_only=True)
w_key = 'deberta.embeddings.word_embeddings.weight'
d_key = 'deberta.embeddings.word_embeddings._weight'
W = sd[w_key].float()
D = sd[d_key].float()
print(f'weight std {W.std():.4f} | _weight std {D.std():.4f} | sum std {(W+D).std():.4f}')
print(f'cos(weight, _weight) row-mean: {F.cosine_similarity(W[:5000], D[:5000]).mean():+.3f}')
print(f'model `weight` == checkpoint weight: {(vide_weight - W).abs().max() < 1e-4}')

gdes = {'weight (loaded)': W, '_weight (delta)': D, 'weight+_weight': W + D}
for name, emb in gdes.items():
    print(f'\n-- ViDeBERTa {name} --')
    dx.donor_space_soundness(emb, vide_vocab, ft_vi, n=1000, k=10)


weight std 0.1276 | _weight std 0.1053 | sum std 0.2211
cos(weight, _weight) row-mean: +0.855
model `weight` == checkpoint weight: True

-- ViDeBERTa weight (loaded) --
── Donor-space soundness (n=1000, k=10, chance≈0.010) ──
  kNN overlap vs FastText: raw 0.013 | syllable-mean 0.013
  mean-centered          : raw 0.013 | syllable-mean 0.012
  verdict (best variant) : DONOR BROKEN — ~chance vs FastText; swap donor   (gates: >0.2 usable, <0.05 broken)

-- ViDeBERTa _weight (delta) --
── Donor-space soundness (n=1000, k=10, chance≈0.010) ──
  kNN overlap vs FastText: raw 0.012 | syllable-mean 0.011
  mean-centered          : raw 0.014 | syllable-mean 0.013
  verdict (best variant) : DONOR BROKEN — ~chance vs FastText; swap donor   (gates: >0.2 usable, <0.05 broken)

-- ViDeBERTa weight+_weight --
── Donor-space soundness (n=1000, k=10, chance≈0.010) ──
  kNN overlap vs FastText: raw 0.013 | syllable-mean 0.013
  mean-centered          : raw 0.014 | syllable-mean 0.013
  verdict (best var

## B. Spearman correlation — the sensitive test
Over ~40k random word pairs, does donor cosine track FastText cosine? Detects diffuse
structure kNN-overlap misses. Run for the best ViDeBERTa GDES variant, PhoBERT, and NeoBERT-EN.


In [6]:
print('NeoBERT-EN positive control:')
import os, gzip, shutil, urllib.request
if not os.path.exists(FT_EN):
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz', FT_EN + '.gz')
    with gzip.open(FT_EN + '.gz', 'rb') as fi, open(FT_EN, 'wb') as fo: shutil.copyfileobj(fi, fo)
    os.remove(FT_EN + '.gz')
ft_en = fasttext.load_model(FT_EN)
corr_neo = da.embedding_semantic_correlation(neo_emb, neo_vocab, ft_en, 'neobert')
del ft_en

print('ViDeBERTa weight:')
corr_vide_w = da.embedding_semantic_correlation(W, vide_vocab, ft_vi, 'videberta')
print('ViDeBERTa weight+_weight:')
corr_vide_s = da.embedding_semantic_correlation(W + D, vide_vocab, ft_vi, 'videberta')
print('PhoBERT:')
corr_pho = da.embedding_semantic_correlation(pho_emb, pho_vocab, ft_vi, 'phobert')


NeoBERT-EN positive control:


  semantic correlation (pairs 39,983): Spearman +0.377 | Pearson +0.449 -> SEMANTIC — donor cos tracks FastText
ViDeBERTa weight:
  semantic correlation (pairs 39,983): Spearman +0.037 | Pearson +0.039 -> NO SEMANTIC SIGNAL — donor cos ⟂ FastText
ViDeBERTa weight+_weight:
  semantic correlation (pairs 39,983): Spearman +0.037 | Pearson +0.037 -> NO SEMANTIC SIGNAL — donor cos ⟂ FastText
PhoBERT:
  semantic correlation (pairs 39,983): Spearman +0.001 | Pearson -0.018 -> NO SEMANTIC SIGNAL — donor cos ⟂ FastText


## C. Related-vs-random cosine gap (interpretable)

In [7]:
print('ViDeBERTa weight:')
gap_vide = da.related_vs_random_gap(W, vide_vocab, ft_vi, 'videberta')
print('ViDeBERTa weight+_weight:')
gap_vide_s = da.related_vs_random_gap(W + D, vide_vocab, ft_vi, 'videberta')
print('PhoBERT:')
gap_pho = da.related_vs_random_gap(pho_emb, pho_vocab, ft_vi, 'phobert')
print('NeoBERT (sanity, vs FastText-vi — expect ~0, EN model on VI words):')
gap_neo = da.related_vs_random_gap(neo_emb, neo_vocab, ft_vi, 'neobert')


ViDeBERTa weight:
  related-vs-random donor cosine: related 0.075 | random 0.071 | gap +0.005 (no locality)
ViDeBERTa weight+_weight:
  related-vs-random donor cosine: related 0.050 | random 0.048 | gap +0.002 (no locality)
PhoBERT:
  related-vs-random donor cosine: related 0.682 | random 0.659 | gap +0.023 (no locality)
NeoBERT (sanity, vs FastText-vi — expect ~0, EN model on VI words):
  related-vs-random donor cosine: related 0.099 | random 0.036 | gap +0.063 (semantic locality present)


## D. Verdict vs predictions

In [8]:
print('=' * 70); print('DONOR STATIC-EMBEDDING VALIDATION — VERDICT'); print('=' * 70)
print(f"B  Spearman  NeoBERT-EN(control) {corr_neo['spearman']:+.3f} | "
      f"ViDeBERTa.w {corr_vide_w['spearman']:+.3f} | ViDeBERTa.w+_w {corr_vide_s['spearman']:+.3f} | "
      f"PhoBERT {corr_pho['spearman']:+.3f}")
print(f"C  gap       ViDeBERTa.w {gap_vide['gap']:+.3f} | ViDeBERTa.w+_w {gap_vide_s['gap']:+.3f} | "
      f"PhoBERT {gap_pho['gap']:+.3f} | NeoBERT/vi {gap_neo['gap']:+.3f}")
best_vide = max(corr_vide_w['spearman'], corr_vide_s['spearman'])
print('-' * 70)
if corr_neo['spearman'] < 0.15:
    print('CONTROL WEAK — even the sensitive metric barely fires on a known-good donor;')
    print('do not over-interpret donor differences. Reconsider whether STATIC-embedding')
    print('semantics is the right SALT criterion at all.')
elif best_vide > 0.2:
    print('ViDeBERTa static embeddings ARE semantic (sensitive test) — earlier kNN~chance')
    print('was the metric being harsh. ViDeBERTa stays the donor; revisit the projection/')
    print('selection pipeline (LOO) as the real leak.')
elif best_vide < 0.05 and corr_pho['spearman'] > 0.15:
    print('CONFIRMED by 2 independent tests: ViDeBERTa static embeddings carry no FastText-')
    print('aligned semantics (DeBERTa-v3 RTD property) while PhoBERT does. This is a real,')
    print('reportable donor limitation for SALT. Options: PhoBERT donor, OR extract a')
    print('DIFFERENT ViDeBERTa representation (contextual / output emb) — decide next.')
else:
    print('Mixed signal — inspect the numbers; no clean verdict.')
print('=' * 70)


DONOR STATIC-EMBEDDING VALIDATION — VERDICT
B  Spearman  NeoBERT-EN(control) +0.377 | ViDeBERTa.w +0.037 | ViDeBERTa.w+_w +0.037 | PhoBERT +0.001
C  gap       ViDeBERTa.w +0.005 | ViDeBERTa.w+_w +0.002 | PhoBERT +0.023 | NeoBERT/vi +0.063
----------------------------------------------------------------------
Mixed signal — inspect the numbers; no clean verdict.
